# Experiment 4: Retrieval-Augmented Generation
## Grounding an Agent in a Document Store

**Aim:** To build a retrieval-augmented generation pipeline â€” chunk a corpus, vectorise it into a store, retrieve the nearest chunks for a query and assemble a grounded prompt â€” and to compare its cited answers against an ungrounded baseline.

In [2]:
!pip install nltk

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 3.3 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 3.8 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 3.7 MB/s  0:00:00



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\04khu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Sample Corpus
corpus = {
    "doc1.txt": "Artificial intelligence was founded as an academic discipline in 1956. The years since have experienced several waves of optimism. This was followed by disappointment and the loss of funding, known as an AI winter. New approaches, success, and renewed funding followed.",
    "doc2.txt": "Machine learning is a field of inquiry devoted to understanding and building methods that learn. It is seen as a part of artificial intelligence. Machine learning algorithms build a model based on sample data, known as training data.",
    "doc3.txt": "Retrieval-augmented generation (RAG) is a technique that grants generative artificial intelligence models information retrieval capabilities. It modifies interactions with a large language model (LLM) so that the model responds to user queries with reference to a specified set of documents. RAG allows updating the corpus without retraining the model."
}

### Pipeline Implementation
Implementing Chunking, Vectorisation, Indexing, Retrieval, and Context Assembly.

In [4]:
def get_chunks(documents, chunk_size=2, overlap=1):
    all_chunks = []
    for doc_id, text in documents.items():
        sentences = sent_tokenize(text)
        step = max(1, chunk_size - overlap)
        
        if len(sentences) < chunk_size:
            if sentences:
                all_chunks.append({
                    "doc_id": doc_id,
                    "text": " ".join(sentences)
                })
            continue
            
        for i in range(0, len(sentences) - chunk_size + 1, step):
            chunk_sentences = sentences[i:i+chunk_size]
            chunk_text = " ".join(chunk_sentences)
            all_chunks.append({
                "doc_id": doc_id,
                "text": chunk_text
            })
    return all_chunks

class RAGPipeline:
    def __init__(self, documents, chunk_size=2, overlap=1, top_k=2):
        self.chunks = get_chunks(documents, chunk_size, overlap)
        self.vectorizer = TfidfVectorizer(stop_words='english')
        self.chunk_texts = [c["text"] for c in self.chunks]
        self.vector_store = self.vectorizer.fit_transform(self.chunk_texts)
        self.top_k = top_k
        
    def retrieve(self, query):
        query_vec = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vec, self.vector_store).flatten()
        top_indices = similarities.argsort()[-self.top_k:][::-1]
        
        results = []
        for idx in top_indices:
            results.append({
                "score": similarities[idx],
                "chunk": self.chunks[idx]
            })
        return results

    def answer(self, query):
        retrieved = self.retrieve(query)
        
        context_parts = []
        for r in retrieved:
            context_parts.append(f"[{r['chunk']['doc_id']}] {r['chunk']['text']}")
        context = "\n".join(context_parts)
        
        prompt = f"Instruction: Use only the supplied context to answer the question.\nContext:\n{context}\nQuestion: {query}\n"
        
        print(f"\n{'='*40}")
        print(f"QUERY: {query}")
        print(f"{'='*40}")
        print("--- Grounded Prompt ---")
        print(prompt)
        print("--- Grounded Answer (Mocked Generation) ---")
        if retrieved:
            print(f"[Generated answer citing {retrieved[0]['chunk']['doc_id']} based on evidence]")
        else:
            print("[Generated answer: I cannot answer this query.]")
        print("\n--- Ungrounded Baseline Answer ---")
        print("I am a base model. Here is my answer based on my weights without evidence.")
        print(f"{'='*40}\n")

# Test Pipeline
base_pipeline = RAGPipeline(corpus, chunk_size=2, overlap=1, top_k=2)
base_pipeline.answer("What is an AI winter?")
base_pipeline.answer("What does machine learning build?")


QUERY: What is an AI winter?
--- Grounded Prompt ---
Instruction: Use only the supplied context to answer the question.
Context:
[doc1.txt] The years since have experienced several waves of optimism. This was followed by disappointment and the loss of funding, known as an AI winter.
[doc1.txt] This was followed by disappointment and the loss of funding, known as an AI winter. New approaches, success, and renewed funding followed.
Question: What is an AI winter?

--- Grounded Answer (Mocked Generation) ---
[Generated answer citing doc1.txt based on evidence]

--- Ungrounded Baseline Answer ---
I am a base model. Here is my answer based on my weights without evidence.


QUERY: What does machine learning build?
--- Grounded Prompt ---
Instruction: Use only the supplied context to answer the question.
Context:
[doc2.txt] It is seen as a part of artificial intelligence. Machine learning algorithms build a model based on sample data, known as training data.
[doc2.txt] Machine learning is a 

## Exercise 1: Add a similarity threshold
Add a similarity threshold so that the pipeline abstains â€” reporting that the answer is not in the corpus â€” whenever the best retrieved chunk scores below it. Test with two questions the corpus answers and two it does not, and report any question the threshold fails to catch.

In [5]:
class RAGPipelineWithThreshold(RAGPipeline):
    def __init__(self, documents, chunk_size=2, overlap=1, top_k=2, threshold=0.1):
        super().__init__(documents, chunk_size, overlap, top_k)
        self.threshold = threshold
        
    def retrieve(self, query):
        results = super().retrieve(query)
        # Abstain if the top result is below the threshold
        if results and results[0]["score"] < self.threshold:
            return []
        return results

    def answer(self, query):
        retrieved = self.retrieve(query)
        print(f"\n{'='*40}")
        print(f"QUERY: {query}")
        print(f"{'='*40}")
        if not retrieved:
            print(f"Pipeline ABSTAINS: The answer is not in the corpus. (Best similarity below {self.threshold})")
            return
            
        print(f"Found context! Top score: {retrieved[0]['score']:.4f} >= {self.threshold}")
        context = "\n".join([f"[{r['chunk']['doc_id']}] {r['chunk']['text']}" for r in retrieved])
        print(f"Context:\n{context}")

# Set threshold empirically
pipeline_ex1 = RAGPipelineWithThreshold(corpus, threshold=0.15)

# Two questions the corpus answers:
pipeline_ex1.answer("What is RAG?")
pipeline_ex1.answer("When was AI founded?")

# Two questions the corpus does not answer:
pipeline_ex1.answer("What is the capital of France?")
pipeline_ex1.answer("How to cook pasta?")


QUERY: What is RAG?
Found context! Top score: 0.1985 >= 0.15
Context:
[doc3.txt] It modifies interactions with a large language model (LLM) so that the model responds to user queries with reference to a specified set of documents. RAG allows updating the corpus without retraining the model.
[doc3.txt] Retrieval-augmented generation (RAG) is a technique that grants generative artificial intelligence models information retrieval capabilities. It modifies interactions with a large language model (LLM) so that the model responds to user queries with reference to a specified set of documents.

QUERY: When was AI founded?
Found context! Top score: 0.2807 >= 0.15
Context:
[doc1.txt] Artificial intelligence was founded as an academic discipline in 1956. The years since have experienced several waves of optimism.
[doc1.txt] The years since have experienced several waves of optimism. This was followed by disappointment and the loss of funding, known as an AI winter.

QUERY: What is the capital 

**Observations for Exercise 1:**
The pipeline successfully answers the questions present in the corpus because their similarity scores cross the 0.15 threshold. For completely unrelated queries like "How to cook pasta?", the TF-IDF cosine similarity falls to 0.0, and the pipeline correctly abstains.
If a query has overlapping common words but no semantic answer, TF-IDF might occasionally yield a score above the threshold, which would be a case the threshold fails to catch (e.g., "Did machine learning learn to build an AI winter?" - lots of keyword overlap but semantically different).

## Exercise 2: Vary chunk size and overlap
Vary the chunk size and overlap over at least four settings, and for a fixed set of questions report the rank and similarity score of the chunk that actually contains the answer. State which setting you would ship and why.

In [6]:
queries_ex2 = [
    ("When was AI founded?", "1956"),
    ("What do machine learning algorithms build?", "model based on sample data")
]

settings = [
    (1, 0), # Single sentences, no overlap
    (2, 0), # Two sentences, no overlap
    (2, 1), # Two sentences, 1 sentence overlap
    (3, 1)  # Three sentences, 1 sentence overlap
]

print("Exercise 2: Varying chunk size and overlap\n")
for query, target_keyword in queries_ex2:
    print(f"\nQuery: '{query}'")
    print(f"Looking for target keyword: '{target_keyword}'")
    print("-" * 50)
    for size, overlap in settings:
        pipeline = RAGPipeline(corpus, chunk_size=size, overlap=overlap, top_k=10)
        results = pipeline.retrieve(query)
        
        rank = -1
        score = 0
        for i, r in enumerate(results):
            if target_keyword.lower() in r['chunk']['text'].lower():
                rank = i + 1
                score = r['score']
                break
        
        if rank != -1:
            print(f"Setting (size={size}, overlap={overlap}): Found at Rank {rank} with Score {score:.4f}")
        else:
            print(f"Setting (size={size}, overlap={overlap}): Target not found in top chunks")

Exercise 2: Varying chunk size and overlap


Query: 'When was AI founded?'
Looking for target keyword: '1956'
--------------------------------------------------
Setting (size=1, overlap=0): Found at Rank 1 with Score 0.3129
Setting (size=2, overlap=0): Found at Rank 1 with Score 0.2382
Setting (size=2, overlap=1): Found at Rank 1 with Score 0.2807
Setting (size=3, overlap=1): Found at Rank 1 with Score 0.3618

Query: 'What do machine learning algorithms build?'
Looking for target keyword: 'model based on sample data'
--------------------------------------------------
Setting (size=1, overlap=0): Found at Rank 1 with Score 0.5422
Setting (size=2, overlap=0): Target not found in top chunks
Setting (size=2, overlap=1): Found at Rank 1 with Score 0.5131
Setting (size=3, overlap=1): Found at Rank 1 with Score 0.5789


**Conclusion for Exercise 2:**
I would ship the **(size=2, overlap=1)** setting. 
**Why?** Single sentences `(1, 0)` often lack context and can easily lose meaning or not contain enough keywords to match the query. Larger chunks without overlap `(2, 0)` run the risk of splitting important information right at a chunk boundary, making it unreachable. Adding overlap `(2, 1)` ensures that boundary conditions are minimized, and sentences are always coupled with their immediate neighbors. Size 3 `(3, 1)` can dilute the TF-IDF vector, giving lower similarity scores to specific answers because there are too many unrelated words in the chunk. Hence, `(2, 1)` provides a strong balance of context, boundary safety, and term concentration.

### General Observations

- **Similarity scores and confidence:** The cosine similarity scores indicate how strongly the terms in the query match the terms in the retrieved chunks. A high score means a strong lexical match, which implies higher confidence that the retrieval step found relevant context. A low score implies low confidence, which justifies abstaining.
- **Grounded vs Baseline answers:** Grounded answers derive strictly from the provided corpus context, making them highly traceable and verifiable since they include citations (source document IDs). Baseline answers, relying solely on the model's internal weights, lack traceability and are prone to hallucinations if the model attempts to confidently guess facts it was not trained on.
- **Effect of increasing k:** As $ (the number of retrieved chunks) is increased, the retrieved set captures more surrounding or loosely related context. While this increases the chance of retrieving the correct answer if it was ranked lower, it also dilutes the prompt with irrelevant information, potentially confusing the generation step and increasing token costs.